# Mr.Shop — Track C: Partner Onboarding & Recommendation Matching
For each user, we score every partner in the catalog on three signals:
1. **Budget fit** (40%) — does the partner's price range overlap the user's stated budget?
2. **Style match** (40%) — how much do the user's style tags overlap the partner's tags?
3. **Purchase history** (20%) — does the partner's tags align with what the user has bought before?

Each score is 0–1, combined into a weighted final score (0–100). We return the top matches
with a plain-language "why" explanation, written the way it would actually appear inside an
Instagram DM.

## 1. Setup — load sample data

In [1]:
import json

WEIGHTS = {
    "budget": 0.4,
    "style": 0.4,
    "history": 0.2,
}

with open("data/users.json") as f:
    users = json.load(f)

with open("data/partners.json") as f:
    partners = json.load(f)

print(f"Loaded {len(users)} users and {len(partners)} partners")

Loaded 4 users and 9 partners


## 2. Preview the sample data

Run this to sanity-check what a user profile and a partner entry look like before scoring.

In [2]:
print("Sample user:")
print(json.dumps(users[0], indent=2))
print("\nSample partner:")
print(json.dumps(partners[0], indent=2))

Sample user:
{
  "user_id": "U1",
  "name": "Priya",
  "style_tags": [
    "streetwear",
    "casual"
  ],
  "budget_min": 2000,
  "budget_max": 6000,
  "wardrobe": [
    "denim jacket",
    "white sneakers",
    "oversized tees"
  ],
  "past_purchase_tags": [
    "streetwear",
    "budget"
  ]
}

Sample partner:
{
  "partner_id": "P1",
  "name": "UrbanEdge",
  "type": "product",
  "category": "streetwear brand",
  "price_min": 2000,
  "price_max": 5000,
  "style_tags": [
    "streetwear",
    "casual"
  ]
}


## 3. Scoring functions

In [3]:
def budget_score(user, partner):
    """1.0 if price ranges fully overlap within user's budget, scaled down for partial overlap, 0 if no overlap."""
    lo = max(user["budget_min"], partner["price_min"])
    hi = min(user["budget_max"], partner["price_max"])
    if hi < lo:
        return 0.0
    overlap = hi - lo
    partner_span = max(partner["price_max"] - partner["price_min"], 1)
    return min(overlap / partner_span, 1.0)


def tag_overlap_score(tags_a, tags_b):
    """Jaccard-style overlap between two tag lists."""
    set_a, set_b = set(tags_a), set(tags_b)
    if not set_a or not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)


def style_score(user, partner):
    return tag_overlap_score(user["style_tags"], partner["style_tags"])


def history_score(user, partner):
    return tag_overlap_score(user["past_purchase_tags"], partner["style_tags"])

## 4. Combine into a weighted match score

In [4]:
def match_score(user, partner):
    b = budget_score(user, partner)
    s = style_score(user, partner)
    h = history_score(user, partner)
    total = (b * WEIGHTS["budget"] + s * WEIGHTS["style"] + h * WEIGHTS["history"]) * 100
    return round(total, 1), {"budget": round(b, 2), "style": round(s, 2), "history": round(h, 2)}


def explain(user, partner, breakdown):
    reasons = []
    if breakdown["budget"] > 0.5:
        reasons.append("fits your \u20b9" + str(user["budget_min"]) + "-" + str(user["budget_max"]) + " budget")
    if breakdown["style"] > 0.3:
        shared = set(user["style_tags"]) & set(partner["style_tags"])
        if shared:
            reasons.append("matches your " + ", ".join(shared) + " style")
    if breakdown["history"] > 0.3:
        reasons.append("similar to what you've bought before")
    if not reasons:
        reasons.append("a possible fit based on your profile")
    return ", ".join(reasons)

## 5. Rank partners for a single user

Try this on one user before running the full demo, so you can see the mechanics clearly.

In [5]:
def top_matches_for_user(user, partners, top_n=3):
    scored = []
    for partner in partners:
        score, breakdown = match_score(user, partner)
        scored.append((score, partner, breakdown))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:top_n], scored

# Try it on the first user
sample_user = users[0]
top, all_scored = top_matches_for_user(sample_user, partners, top_n=3)

print("Top matches for " + sample_user["name"] + ":\n")
for rank, (score, partner, breakdown) in enumerate(top, start=1):
    print(str(rank) + ". " + partner["name"] + " (" + partner["category"] + ") - score " + str(score) + "/100")
    print("   breakdown -> " + str(breakdown))
    print("   why: " + explain(sample_user, partner, breakdown) + "\n")

Top matches for Priya:

1. UrbanEdge (streetwear brand) - score 86.7/100
   breakdown -> {'budget': 1.0, 'style': 1.0, 'history': 0.33}
   why: fits your ₹2000-6000 budget, matches your casual, streetwear style, similar to what you've bought before

2. StreetLane (budget streetwear brand) - score 53.3/100
   breakdown -> {'budget': 0.5, 'style': 0.33, 'history': 1.0}
   why: matches your streetwear style, similar to what you've bought before

3. Zoya Khan (personal stylist) - score 46.7/100
   breakdown -> {'budget': 0.0, 'style': 1.0, 'history': 0.33}
   why: matches your casual, streetwear style, similar to what you've bought before



## 6. Full demo — run against every user

This reproduces the complete output, including a deliberately-shown low-scoring match (to prove the logic discriminates) and a sample Instagram-DM-style reply.

In [6]:
def run_demo():
    for user in users:
        print("=" * 70)
        print("USER: " + user["name"] + "  (style: " + ", ".join(user["style_tags"]) +
              ", budget: \u20b9" + str(user["budget_min"]) + "-" + str(user["budget_max"]) + ")")
        print("=" * 70)

        top, all_scored = top_matches_for_user(user, partners, top_n=3)

        print("\nTop matches:")
        for rank, (score, partner, breakdown) in enumerate(top, start=1):
            reason = explain(user, partner, breakdown)
            print("  " + str(rank) + ". " + partner["name"] + " (" + partner["category"] + ") - score " + str(score) + "/100")
            print("     breakdown -> budget:" + str(breakdown["budget"]) + " style:" + str(breakdown["style"]) + " history:" + str(breakdown["history"]))
            print("     why: " + reason)

        lowest = min(all_scored, key=lambda x: x[0])
        print("\n  Lowest-scoring (correctly filtered out): " + lowest[1]["name"] + " - score " + str(lowest[0]) + "/100")

        best_score, best_partner, best_breakdown = top[0]
        reason = explain(user, best_partner, best_breakdown)
        print("\n  --- Sample Instagram DM reply ---")
        print('  "Hey ' + user["name"] + "! Based on your wardrobe and budget, I think you'd love " +
              best_partner["name"] + " - it " + reason + '. Want me to show you a few pieces? \U0001F457"')
        print()

run_demo()

USER: Priya  (style: streetwear, casual, budget: ₹2000-6000)

Top matches:
  1. UrbanEdge (streetwear brand) - score 86.7/100
     breakdown -> budget:1.0 style:1.0 history:0.33
     why: fits your ₹2000-6000 budget, matches your casual, streetwear style, similar to what you've bought before
  2. StreetLane (budget streetwear brand) - score 53.3/100
     breakdown -> budget:0.5 style:0.33 history:1.0
     why: matches your streetwear style, similar to what you've bought before
  3. Zoya Khan (personal stylist) - score 46.7/100
     breakdown -> budget:0.0 style:1.0 history:0.33
     why: matches your casual, streetwear style, similar to what you've bought before

  Lowest-scoring (correctly filtered out): Formalcraft - score 0.0/100

  --- Sample Instagram DM reply ---
  "Hey Priya! Based on your wardrobe and budget, I think you'd love UrbanEdge - it fits your ₹2000-6000 budget, matches your casual, streetwear style, similar to what you've bought before. Want me to show you a few piece

## 7. Try your own scenario

Change the values below to test edge cases live during your presentation — e.g. a user with
a very narrow budget, or conflicting style tags.

In [7]:
custom_user = {
    "user_id": "TEST",
    "name": "Test User",
    "style_tags": ["streetwear", "athleisure"],
    "budget_min": 3000,
    "budget_max": 7000,
    "wardrobe": ["hoodie", "sneakers"],
    "past_purchase_tags": ["sportswear"]
}

top, _ = top_matches_for_user(custom_user, partners, top_n=3)
for rank, (score, partner, breakdown) in enumerate(top, start=1):
    print(str(rank) + ". " + partner["name"] + " - score " + str(score) + "/100 - " + explain(custom_user, partner, breakdown))

1. UrbanEdge - score 40.0/100 - fits your ₹3000-7000 budget, matches your streetwear style
2. ActiveFit - score 37.3/100 - fits your ₹3000-7000 budget, matches your athleisure style
3. StreetLane - score 13.3/100 - matches your streetwear style
